In [2]:
import sys
import pandas as pd
import numpy as np
sys.path.append('..')  # Si estás en notebooks/

from src.data.data_quality import analyze_data_quality, show_categorical_columns    

In [3]:
df = pd.read_csv('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/raw/items_titles.csv', sep=',', encoding='utf-8')

In [8]:
df2 = pd.read_csv('C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/raw/items_titles_test.csv', sep=',', encoding='utf-8')

In [4]:
df.shape

(30000, 1)

In [6]:
df.shape

(30000, 1)

In [13]:
df2.shape

(10000, 5)

In [7]:
df.head()

,ITE_ITEM_TITLE
0,Tênis Ascension Posh Masculino - Preto E Verme...
1,Tenis Para Caminhada Super Levinho Spider Corr...
2,Tênis Feminino Le Parc Hocks Black/ice Origina...
3,Tênis Olympikus Esportivo Academia Nova Tendên...
4,Inteligente Led Bicicleta Tauda Luz Usb Bicicl...


In [9]:
df2.head()

,ITE_ITEM_TITLE
0,Tênis Olympikus Esporte Valente - Masculino Kids
1,Bicicleta Barra Forte Samy C/ 6 Marchas Cubo C...
2,Tênis Usthemp Slip-on Temático - Labrador 2
3,Tênis Casual Feminino Moleca Tecido Tie Dye
4,Tênis Star Baby Sapatinho Conforto + Brinde


In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import ks_2samp
import time

# ============================================================================
# 1. PREPARACIÓN Y LIMPIEZA DE TEXTOS
# ============================================================================

def clean_text(text):
    """Limpia y normaliza texto para mejor vectorización"""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Remover caracteres especiales pero mantener palabras
    text = ''.join(c if c.isalnum() or c.isspace() else ' ' for c in text)
    # Remover espacios múltiples
    text = ' '.join(text.split())
    return text

# Limpiar textos
df['title_clean'] = df['ITE_ITEM_TITLE'].apply(clean_text)
df2['title_clean'] = df2['ITE_ITEM_TITLE'].apply(clean_text)

print(f"Corpus entrenamiento: {len(df)} títulos")
print(f"Corpus test: {len(df2)} títulos")
print("\nEjemplos limpios:")
print(df['title_clean'].head(3).tolist())

Corpus entrenamiento: 30000 títulos
Corpus test: 10000 títulos

Ejemplos limpios:
['tênis ascension posh masculino preto e vermelho', 'tenis para caminhada super levinho spider corrida', 'tênis feminino le parc hocks black ice original envio já']


In [11]:
# ============================================================================
# 2. VECTORIZACIÓN CON TF-IDF
# ============================================================================

print("\n" + "="*80)
print("VECTORIZACIÓN TF-IDF")
print("="*80)

# Entrenar vectorizador en df (corpus de entrenamiento)
start_time = time.time()

vectorizer = TfidfVectorizer(
    max_features=5000,      # Top 5000 palabras más importantes
    ngram_range=(1, 3),     # Unigramas, bigramas y trigramas
    min_df=2,               # Palabra debe aparecer en al menos 2 documentos
    max_df=0.8,             # Palabra no debe aparecer en más del 80% de docs
    strip_accents='unicode',
    token_pattern=r'\b\w+\b'
)

# Fit en df (entrenamiento)
tfidf_train = vectorizer.fit_transform(df['title_clean'])

# Transform en df2 (test)
tfidf_test = vectorizer.transform(df2['title_clean'])

vectorization_time = time.time() - start_time

print(f"✅ Vectorización completada en {vectorization_time:.2f} segundos")
print(f"Vocabulario: {len(vectorizer.vocabulary_)} términos")
print(f"Shape train: {tfidf_train.shape}")
print(f"Shape test: {tfidf_test.shape}")
print(f"Sparsity train: {(1 - tfidf_train.nnz / (tfidf_train.shape[0] * tfidf_train.shape[1])) * 100:.2f}%")


VECTORIZACIÓN TF-IDF
✅ Vectorización completada en 2.90 segundos
Vocabulario: 5000 términos
Shape train: (30000, 5000)
Shape test: (10000, 5000)
Sparsity train: 99.80%


In [12]:
# ============================================================================
# 3. CÁLCULO DE SIMILARIDAD DE COSENO
# ============================================================================

print("\n" + "="*80)
print("SIMILARIDAD DE COSENO")
print("="*80)

# Calcular similaridad coseno entre test y train
start_time = time.time()

# Similaridad de cada título en df2 con todos los títulos en df
cosine_sim_matrix = cosine_similarity(tfidf_test, tfidf_train)

cosine_time = time.time() - start_time

print(f"✅ Similaridad de coseno calculada en {cosine_time:.2f} segundos")
print(f"Shape matriz similaridad: {cosine_sim_matrix.shape}")
print(f"(cada fila = título de test, cada columna = título de train)")

# Para cada título en df2, encontrar el más similar en df
df2['best_match_idx'] = cosine_sim_matrix.argmax(axis=1)
df2['best_match_score'] = cosine_sim_matrix.max(axis=1)
df2['best_match_title'] = df.iloc[df2['best_match_idx']]['ITE_ITEM_TITLE'].values

# Estadísticas de similaridad
print(f"\n📊 ESTADÍSTICAS DE SIMILARIDAD:")
print(f"Similaridad promedio: {df2['best_match_score'].mean():.4f}")
print(f"Similaridad mediana: {df2['best_match_score'].median():.4f}")
print(f"Similaridad mínima: {df2['best_match_score'].min():.4f}")
print(f"Similaridad máxima: {df2['best_match_score'].max():.4f}")
print(f"Desviación estándar: {df2['best_match_score'].std():.4f}")

# Mostrar algunos ejemplos
print(f"\n🔍 TOP 5 MEJORES MATCHES:")
top_matches = df2.nlargest(5, 'best_match_score')[['ITE_ITEM_TITLE', 'best_match_title', 'best_match_score']]
for idx, row in top_matches.iterrows():
    print(f"\nTest: {row['ITE_ITEM_TITLE'][:60]}...")
    print(f"Match: {row['best_match_title'][:60]}...")
    print(f"Score: {row['best_match_score']:.4f}")


SIMILARIDAD DE COSENO
✅ Similaridad de coseno calculada en 8.77 segundos
Shape matriz similaridad: (10000, 30000)
(cada fila = título de test, cada columna = título de train)

📊 ESTADÍSTICAS DE SIMILARIDAD:
Similaridad promedio: 0.7848
Similaridad mediana: 0.7804
Similaridad mínima: 0.0000
Similaridad máxima: 1.0000
Desviación estándar: 0.1662

🔍 TOP 5 MEJORES MATCHES:

Test: Tenis Masculino Preto Para Corrida Sem Cadarço 7mboots...
Match: Tenis Masculino Preto Para Corrida Sem Cadarço 7mboots...
Score: 1.0000

Test: Tv 50 Polegadas Philco Smart...
Match: Tv 50 Polegadas Philco Smart...
Score: 1.0000

Test: Tênis Nike Dunk High Michigan...
Match: Tenis Nike Dunk High Michigan...
Score: 1.0000

Test: Promoção Kit 3 Sapatenis Carteira, Cinto E Boné De Brinde ...
Match: Promoção Kit 3 Sapatenis Carteira, Cinto E Boné De Brinde ...
Score: 1.0000

Test: Kit 2 Tênis Escolar Masculino Macio Sem Cadarço Novidade...
Match: Kit 2 Tênis Escolar Masculino Macio Sem Cadarço Novidade...
Score: 1.00

In [16]:
# ============================================================================
# GENERAR PARES EN DF2 (TEST) CON VECTORIZADOR ENTRENADO EN DF
# ============================================================================

print("\n" + "="*80)
print("GENERACIÓN DE PARES EN TEST SET")
print("="*80)

# Transformar df2 con el vectorizador ya entrenado en df
start_transform = time.time()
tfidf_test = vectorizer.transform(df2['title_clean'])
transform_time = time.time() - start_transform

print(f"✅ Transform completado en {transform_time:.2f}s")
print(f"Shape test: {tfidf_test.shape}")

# Calcular matriz de similaridad entre todos los títulos de test
start_similarity = time.time()
cosine_sim_matrix = cosine_similarity(tfidf_test)
similarity_time = time.time() - start_similarity

print(f"✅ Similaridad calculada en {similarity_time:.2f}s")
print(f"Shape matriz: {cosine_sim_matrix.shape}")

# Generar lista de pares
start_pairs = time.time()
pairs_list = []

n = len(df2)
for i in range(n):
    for j in range(i + 1, n):  # Solo triángulo superior para evitar duplicados
        pairs_list.append({
            'title_1': df2.iloc[i]['ITE_ITEM_TITLE'],
            'title_2': df2.iloc[j]['ITE_ITEM_TITLE'],
            'similarity_score': cosine_sim_matrix[i, j]
        })

pairs_time = time.time() - start_pairs

# Crear DataFrame de pares y ordenar por score descendente
df_pairs = pd.DataFrame(pairs_list)
df_pairs = df_pairs.sort_values('similarity_score', ascending=False).reset_index(drop=True)

print(f"✅ Pares generados en {pairs_time:.2f}s")
print(f"Total de pares: {len(df_pairs):,}")

# Estadísticas
print(f"\n📊 ESTADÍSTICAS:")
print(f"Similaridad promedio: {df_pairs['similarity_score'].mean():.4f}")
print(f"Similaridad mediana: {df_pairs['similarity_score'].median():.4f}")
print(f"Similaridad máxima: {df_pairs['similarity_score'].max():.4f}")
print(f"Similaridad mínima: {df_pairs['similarity_score'].min():.4f}")

# Mostrar top pares
print(f"\n🏆 TOP 20 PARES MÁS SIMILARES:")
print(df_pairs.head(20).to_string(index=False))


GENERACIÓN DE PARES EN TEST SET
✅ Transform completado en 0.31s
Shape test: (10000, 5000)
✅ Similaridad calculada en 2.51s
Shape matriz: (10000, 10000)
✅ Pares generados en 5721.69s
Total de pares: 49,995,000

📊 ESTADÍSTICAS:
Similaridad promedio: 0.0143
Similaridad mediana: 0.0042
Similaridad máxima: 1.0000
Similaridad mínima: 0.0000

🏆 TOP 20 PARES MÁS SIMILARES:
                                            title_1                                                title_2  similarity_score
     Tênis Usthemp Chunky Smooth Tem - Skull Mistic            Tênis Usthemp Chunky Smooth Tem- Groundmade               1.0
                   Sapatênis Masculino Freeway Land                      Sapatênis Masculino Freeway Hallf               1.0
                         Tênis Infantil Menina Zeuz               Tenis Infantil Menina Princesa Cinderela               1.0
        Tenis Feminino Kolosh Mali Preto C0379-0001                      Tênis Feminino Kolosh C2783 Preto               1.0
      

In [17]:
df_pairs.shape

(49995000, 3)

In [18]:
df_pairs.head()

,title_1,title_2,similarity_score
0,Tênis Usthemp Chunky Smooth Tem - Skull Mistic,Tênis Usthemp Chunky Smooth Tem- Groundmade,1.0
1,Sapatênis Masculino Freeway Land,Sapatênis Masculino Freeway Hallf,1.0
2,Tênis Infantil Menina Zeuz,Tenis Infantil Menina Princesa Cinderela,1.0
3,Tenis Feminino Kolosh Mali Preto C0379-0001,Tênis Feminino Kolosh C2783 Preto,1.0
4,Tênis Infantil Menina Zeuz,Tênis Infantil Menina Coroa .,1.0


In [20]:
df_pairs.to_csv('C:/Users/carlo/Downloads/df_pairs.csv', index=False, sep=';')